# NeuroZip byte-architecture sweep on Kaggle GPUs

This notebook is an orchestration layer. It clones the public NeuroZip repository, prepares one deterministic raw-byte WikiText-103 split, trains every configured architecture under the same budget, and benchmarks every checkpoint through the real NeuroZip CDF/range-coder pipeline.

The sweep keeps the existing two-layer GRU as the baseline and adds an LSTM control, a small causal Transformer, Mamba-Lite, Griffin-Lite, Gated DeltaNet-Lite, and Gated DeltaNet-2-Lite. The latter four are pure-PyTorch reference variants with explicit streaming states; their results should be read as implementation-level comparisons, not optimized-kernel claims.

A model is only `PASSED` when decompression is byte-identical and SHA-256-identical. The final recommendation uses actual full-stream BPB, encode/decode speed, and memory relative to the GRU, not validation loss alone.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

REPO_URL = 'https://github.com/Chickaboo/NeuroZip.git'
WORK_ROOT = Path('/kaggle/working/neurozip')
if WORK_ROOT.exists():
    shutil.rmtree(WORK_ROOT)
subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(WORK_ROOT)], check=True)
sys.path.insert(0, str(WORK_ROOT / 'src'))
RUN_ENV = os.environ.copy()
RUN_ENV['PYTHONPATH'] = str(WORK_ROOT / 'src') + os.pathsep + RUN_ENV.get('PYTHONPATH', '')
print('Cloned source:', REPO_URL)
print('Working source:', WORK_ROOT)


In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError('CUDA GPUs are required; enable Kaggle GPUs before running the sweep.')
GPU_COUNT = torch.cuda.device_count()
if GPU_COUNT < 2:
    raise RuntimeError(f'This experiment expects both Kaggle T4s; only {GPU_COUNT} GPU(s) are visible.')
print('PyTorch:', torch.__version__)
print('GPUs:', [torch.cuda.get_device_name(i) for i in range(GPU_COUNT)])

SWEEP_CONFIG = json.loads((WORK_ROOT / 'configs' / 'architecture_sweep_wikitext103_kaggle.json').read_text())
RUN_ROOT = Path('/kaggle/working/neurozip-byte-architecture-sweep')
DATA_ROOT = RUN_ROOT / 'data'
CACHE_ROOT = Path('/kaggle/working/wikitext-103-cache')
if RUN_ROOT.exists():
    shutil.rmtree(RUN_ROOT)
DATA_ROOT.mkdir(parents=True, exist_ok=True)

prepare_cmd = [
    sys.executable, '-m', 'neurozip.data.prepare_wikitext',
    '--output-dir', str(DATA_ROOT),
    '--cache-dir', str(CACHE_ROOT),
    '--url', 'https://huggingface.co/datasets/mattdangerw/wikitext-103-raw/resolve/main/wikitext-103-raw-v1.zip?download=true',
    '--train-bytes', str(SWEEP_CONFIG['dataset']['train_bytes']),
    '--valid-bytes', str(SWEEP_CONFIG['dataset']['validation_bytes']),
    '--seed', str(SWEEP_CONFIG['dataset']['seed']),
]
print('Preparing:', ' '.join(prepare_cmd))
subprocess.run(prepare_cmd, cwd=WORK_ROOT, env=RUN_ENV, check=True)
manifest = json.loads((DATA_ROOT / 'manifest.json').read_text())
print(json.dumps(manifest, indent=2, sort_keys=True))


In [ ]:
from neurozip.models.registry import build_model

# Verify the parameter-matching plan before spending GPU time.
parameter_rows = []
for spec in SWEEP_CONFIG['architectures']:
    model = build_model(spec['architecture'], **spec['args'])
    count = sum(parameter.numel() for parameter in model.parameters())
    parameter_rows.append({
        'architecture': spec['name'],
        'label': spec['label'],
        'parameters': count,
        'delta_vs_target': count - SWEEP_CONFIG['matching']['target_parameter_count'],
    })
print(json.dumps(parameter_rows, indent=2))


In [ ]:
import time

ARCHITECTURE_ROOT = RUN_ROOT / 'architectures'
ARCHITECTURE_ROOT.mkdir(parents=True, exist_ok=True)
training = SWEEP_CONFIG['training']
for spec in SWEEP_CONFIG['architectures']:
    name = spec['name']
    output_dir = ARCHITECTURE_ROOT / name
    if output_dir.exists():
        shutil.rmtree(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    train_cmd = [
        sys.executable, '-m', 'torch.distributed.run',
        '--standalone', '--nproc-per-node', str(GPU_COUNT),
        '-m', 'neurozip.train',
        '--architecture', spec['architecture'],
        '--train-path', str(DATA_ROOT / 'train.raw'),
        '--valid-path', str(DATA_ROOT / 'validation.raw'),
        '--output-dir', str(output_dir),
        '--steps', str(training['steps']),
        '--batch-size', str(training['batch_size_per_gpu']),
        '--sequence-length', str(SWEEP_CONFIG['representation']['sequence_length']),
        '--learning-rate', str(training['learning_rate']),
        '--weight-decay', str(training['weight_decay']),
        '--gradient-clip', str(training['gradient_clip']),
        '--eval-every', str(training['eval_every']),
        '--validation-eval-bytes', str(training['validation_eval_bytes']),
        '--seed', str(SWEEP_CONFIG['dataset']['seed']),
        '--device', training['device'],
    ]
    for key, value in spec['args'].items():
        train_cmd.extend(['--' + key.replace('_', '-'), str(value)])
    print(f'\n===== Training {name} =====')
    print(' '.join(train_cmd))
    started = time.perf_counter()
    subprocess.run(train_cmd, cwd=WORK_ROOT, env=RUN_ENV, check=True)
    print(f'{name} wall time: {time.perf_counter() - started:.1f}s')


In [ ]:
BENCHMARK_ROOT = RUN_ROOT / 'benchmark'
benchmark_cmd = [
    sys.executable, '-m', 'neurozip.experiments.architecture_benchmark',
    '--artifacts-root', str(ARCHITECTURE_ROOT),
    '--input', str(DATA_ROOT / 'validation.raw'),
    '--bytes', str(SWEEP_CONFIG['dataset']['heldout_benchmark_bytes']),
    '--output-dir', str(BENCHMARK_ROOT),
    '--device', 'cuda:0',
    '--cdf-bits', str(SWEEP_CONFIG['coding']['cdf_bits']),
]
print('Benchmarking:', ' '.join(benchmark_cmd))
subprocess.run(benchmark_cmd, cwd=WORK_ROOT, env=RUN_ENV, check=True)
comparison = json.loads((BENCHMARK_ROOT / 'comparison.json').read_text())
print('Recommendation:', comparison['recommendation'])


In [ ]:
rows = comparison['results']
try:
    import pandas as pd
    columns = [
        'architecture', 'status', 'train_loss_nats_per_byte', 'train_bpb',
        'validation_loss_nats_per_byte', 'validation_bpb',
        'actual_compressed_bpb', 'payload_bpb', 'compression_ratio',
        'parameter_count', 'checkpoint_bytes', 'training_wall_time_seconds',
        'training_bytes_per_second', 'encode_bytes_per_second',
        'decode_bytes_per_second', 'training_peak_gpu_memory_bytes',
        'training_peak_cpu_memory_bytes', 'encode_peak_gpu_memory_bytes',
        'decode_peak_gpu_memory_bytes', 'peak_process_rss_bytes',
        'byte_identical', 'sha256_identical', 'exact_round_trip',
        'tradeoff_score',
    ]
    display(pd.DataFrame(rows)[columns])
except ImportError:
    print(json.dumps(rows, indent=2, sort_keys=True))

print('\nGeneration sanity checks:')
for row in rows:
    if 'sample_text' in row:
        print(row['architecture'], repr(row['sample_text'][:160]))


In [ ]:
archive_base = Path('/kaggle/working/neurozip-byte-architecture-sweep-artifacts')
archive_path = shutil.make_archive(str(archive_base), 'zip', root_dir=RUN_ROOT)
print('Download this Kaggle output:', archive_path)
print('Comparison report:', BENCHMARK_ROOT / 'comparison.md')
